## Summary

## 0. Data

In [1]:
import pandas as pd

url = "http://byungwan.com/class/Coupon_Sample.xlsx"
coupon = pd.read_excel(url)

coupon.head()

,CID,Gender,Age,CClass,Discount,Visits,Email,SMS,MStatus,Purchases,Sales
0,1,0,41,2,0,1,0,1,1,1,0.92
1,2,0,41,1,0,0,0,0,0,0,0.00
2,3,0,30,2,0,3,0,0,0,1,1.54
3,4,0,28,2,0,2,0,0,0,2,0.88
4,5,1,24,2,1,5,0,0,0,2,0.53


In [2]:
coupon["Discount"].value_counts()

Discount
0    377
1    122
Name: count, dtype: int64

## 1. Preprocessing 

### 1.1 Balancing (Upsampling / Downsampling)

In [5]:
from sklearn.utils import resample

coupon0 = coupon[coupon["Discount"] == 0]
coupon1 = coupon[coupon["Discount"] == 1]

coupon1_up = resample(coupon1, replace=True, n_samples=377, random_state=0)
coupon_up = pd.concat([coupon0, coupon1_up], axis=0)

coupon_up["Discount"].value_counts()

Discount
0    377
1    377
Name: count, dtype: int64

### 1.2 Feature Selection

In [9]:
cols = coupon_up.columns.tolist()
print(cols)

['CID', 'Gender', 'Age', 'CClass', 'Discount', 'Visits', 'Email', 'SMS', 'MStatus', 'Purchases', 'Sales']


In [14]:
cols1 = [cols[4]] + cols[1:4] + cols[5:]
print(cols1)

['Discount', 'Gender', 'Age', 'CClass', 'Visits', 'Email', 'SMS', 'MStatus', 'Purchases', 'Sales']


In [17]:
coupon2 = coupon_up[cols1]
coupon2.head()

,Discount,Gender,Age,CClass,Visits,Email,SMS,MStatus,Purchases,Sales
0,0,0,41,2,1,0,1,1,1,0.92
1,0,0,41,1,0,0,0,0,0,0.00
2,0,0,30,2,3,0,0,0,1,1.54
3,0,0,28,2,2,0,0,0,2,0.88
6,0,0,30,2,1,0,0,0,1,0.58


In [18]:
y = coupon2.iloc[:, 0]
x = coupon2.iloc[:, 1:]

### 1.3 Scailing

In [21]:
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()
scaler_model = scaler.fit(x)
scaled_x = scaler_model.transform(x)

## 2. Models

In [25]:
from sklearn.model_selection import cross_val_score
import numpy as np

from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.neural_network import MLPClassifier
from sklearn.svm import SVC
from sklearn.linear_model import LogisticRegression

In [28]:
models = []

lr = LogisticRegression()
models.append(("LR", lr))

models

[('LR', LogisticRegression())]

In [29]:
dtc = DecisionTreeClassifier(criterion="entropy", max_depth=5)
models.append(("DT", dtc))

models

[('LR', LogisticRegression()),
 ('DT', DecisionTreeClassifier(criterion='entropy', max_depth=5))]

In [30]:
rfc = RandomForestClassifier(n_estimators=5, random_state=0)
models.append(("RF", rfc))

models

[('LR', LogisticRegression()),
 ('DT', DecisionTreeClassifier(criterion='entropy', max_depth=5)),
 ('RF', RandomForestClassifier(n_estimators=5, random_state=0))]

In [35]:
ann = MLPClassifier(activation="relu", hidden_layer_sizes=(5), max_iter=3000, random_state=0)
models.append(("ANN", ann))

models

[('LR', LogisticRegression()),
 ('DT', DecisionTreeClassifier(criterion='entropy', max_depth=5)),
 ('RF', RandomForestClassifier(n_estimators=5, random_state=0)),
 ('ANN', MLPClassifier(hidden_layer_sizes=2, max_iter=3000, random_state=0)),
 ('ANN', MLPClassifier(hidden_layer_sizes=5, max_iter=3000, random_state=0))]

In [36]:
svm = SVC(kernel="rbf", C=1, gamma=1)
models.append(("SVM", svm))

models

[('LR', LogisticRegression()),
 ('DT', DecisionTreeClassifier(criterion='entropy', max_depth=5)),
 ('RF', RandomForestClassifier(n_estimators=5, random_state=0)),
 ('ANN', MLPClassifier(hidden_layer_sizes=2, max_iter=3000, random_state=0)),
 ('ANN', MLPClassifier(hidden_layer_sizes=5, max_iter=3000, random_state=0)),
 ('SVM', SVC(C=1, gamma=1))]

In [37]:
for name, model in models:
    scores = cross_val_score(model, scaled_x, y, cv=5)
    mean = np.mean(scores)
    std = np.std(scores)
    print(name + ": " + str(mean) + " " + "(" + str(std) +")")

LR: 0.7466666666666667 (0.02678918683831544)
DT: 0.897916114790287 (0.017410788450942637)
RF: 0.9469580573951435 (0.01823500256454366)
ANN: 0.7440176600441502 (0.020942980759659675)
ANN: 0.7506843267108168 (0.05298263423085062)
SVM: 0.8700485651214128 (0.012780236095219557)


### 3. Application

In [38]:
RF = RandomForestClassifier(n_estimators=5, random_state=0)
RF_model = RF.fit(scaled_x, y)

In [39]:
url = "http://byungwan.com/class/Coupon_Test.xlsx"
coupon_test = pd.read_excel(url)
coupon_test.head()

,CID,Gender,Age,CClass,Discount,Visits,Email,SMS,MStatus,Purchases,Sales
0,1,1,34,2,0,1,1,1,0,1,0.51
1,2,1,23,2,0,12,0,0,0,4,1.72
2,3,1,20,1,0,1,0,0,0,0,0.00
3,4,0,17,1,0,3,0,0,0,0,0.00
4,5,0,16,2,0,3,0,0,1,1,0.37


In [40]:
coupon2.head()

,Discount,Gender,Age,CClass,Visits,Email,SMS,MStatus,Purchases,Sales
0,0,0,41,2,1,0,1,1,1,0.92
1,0,0,41,1,0,0,0,0,0,0.00
2,0,0,30,2,3,0,0,0,1,1.54
3,0,0,28,2,2,0,0,0,2,0.88
6,0,0,30,2,1,0,0,0,1,0.58


In [42]:
coupon_test2 = coupon_test[cols1]
coupon_test2.head()

,Discount,Gender,Age,CClass,Visits,Email,SMS,MStatus,Purchases,Sales
0,0,1,34,2,1,1,1,0,1,0.51
1,0,1,23,2,12,0,0,0,4,1.72
2,0,1,20,1,1,0,0,0,0,0.00
3,0,0,17,1,3,0,0,0,0,0.00
4,0,0,16,2,3,0,0,1,1,0.37


In [44]:
y_test = coupon_test2.iloc[:, 0]
x_test = coupon_test2.iloc[:, 1:]

scaled_x_test = scaler_model.transform(x_test)
y_pred = RF_model.predict(scaled_x_test)
print(y_pred)

[0 0 0 0 0 0 0 1 0 0 0 1 1 0 0 0 1 1 0 0 0 0 1 1 1 0 0 0 0 0 0 0 1 1 1 1 0
 0 0 0 0 0 1 0 0 0 0 0 1 0 1 1 0 1 1 0 0 0 1 0 0 0 0 0 1 0 1 1 0 1 0 0 0 0
 0 1 1 1 0 0 0 0 0 1 1 0 1 1 0 1 0 0 1 1 0 0 1 0 0 0 1 0 1 0 0 1 0 1 0 0 1
 1 0 0 1 0 0 0 0 0 1 0 0 1 1 0 1 1 0 0 1 0 1 1 1 0 1 0 1 0 0 0 1 1 0 1 0 1
 1 1 0 0 1 0 1 1 1 1 1 0 1 0 0 1 0 1 0 0 0 0 1 1 1 0 1 0 0 1 0 0 1 0 1 0 0
 1 1 0 0 0 0 0 0 1 1 1 0 0 1 0]


In [45]:
from sklearn import metrics

metrics.accuracy_score(y_test, y_pred)

0.725